# Company HR Vectorless RAG

A small RAG example that does **not** use embeddings or a vector database.

The flow is:

`User Question → Keyword Retrieval → Relevant HR Text → LLM → Answer`

The retrieval step simply checks which HR document contains the important words from the question. LangGraph controls the RAG workflow.

## 1. Install the current packages

Run this once in a fresh environment.

In [ ]:
%pip install -U langchain langchain-openai langgraph

## 2. Imports and LLM setup

Set `OPENAI_API_KEY` in your environment before running the notebook.

In [ ]:
import re
from pathlib import Path
from typing import TypedDict

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
)

## 3. Load the three HR files

There is no document embedding here. We simply read the text files.

In [ ]:
DATA_DIR = Path("company_hr_vectorless_rag")

documents = {}

for file in DATA_DIR.glob("*.txt"):
    documents[file.name] = file.read_text(encoding="utf-8")

print("Loaded files:", list(documents))

## 4. Simple vectorless retriever

The retriever extracts words from the question and counts how many of those words appear in each HR document.

No embedding model is used.

No vector database is used.

This is the key idea of this demo: **retrieve by text/keyword matching instead of semantic vectors**.

In [ ]:
STOP_WORDS = {
    "the", "is", "a", "an", "and", "or", "to", "of", "for", "in",
    "on", "can", "i", "do", "does", "my", "what", "how", "when", "are"
}

def retrieve(question: str, top_k: int = 2):
    words = set(re.findall(r"[a-zA-Z]+", question.lower())) - STOP_WORDS

    scores = []
    for name, text in documents.items():
        text_words = set(re.findall(r"[a-zA-Z]+", text.lower()))
        score = len(words & text_words)
        scores.append((score, name, text))

    scores.sort(reverse=True)
    return [(name, text) for score, name, text in scores[:top_k] if score > 0]

## 5. Test retrieval before adding the LLM

This makes the retrieval step easy to understand.

In [ ]:
question = "How many paid leaves do employees get?"

results = retrieve(question)

for name, text in results:
    print(f"--- {name} ---")
    print(text)

## 6. Define the LangGraph state

The state carries the question, retrieved context, and final answer through the graph.

In [ ]:
class HRState(TypedDict, total=False):
    question: str
    context: str
    answer: str

## 7. Retrieval node

This node performs the vectorless retrieval.

In [ ]:
def retrieve_node(state: HRState):
    results = retrieve(state["question"])
    context = "\n\n".join(
        f"[{name}]\n{text}" for name, text in results
    )
    return {"context": context}

## 8. Answer node

The LLM receives the question plus only the retrieved HR context.

In [ ]:
def answer_node(state: HRState):
    prompt = f"""
You are a company HR assistant.

Answer the question using only the HR context below.
If the answer is not present in the context, say that the HR documents do not provide the answer.

HR context:
{state['context']}

Question:
{state['question']}
"""

    response = llm.invoke(prompt)
    return {"answer": response.content}

## 9. Build the LangGraph RAG pipeline

The graph is intentionally small:

`START → Retrieve → Answer → END`

In [ ]:
graph = StateGraph(HRState)

graph.add_node("retrieve", retrieve_node)
graph.add_node("answer", answer_node)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "answer")
graph.add_edge("answer", END)

app = graph.compile()

## 10. Run the complete Vectorless RAG


In [ ]:
question = "How many days can I work from home each week?"

result = app.invoke({"question": question})

print(result["answer"])

## 11. Try another HR question


In [ ]:
question = "When is salary processed?"

result = app.invoke({"question": question})

print(result["answer"])

## What makes this Vectorless RAG?

Traditional vector RAG usually looks like:

`Documents → Embeddings → Vector Database → Similarity Search → LLM`

This demo uses:

`Documents → Keyword Matching → Relevant Text → LLM`

So the retrieval idea is still present, but there is no embedding model and no vector database.

This is useful for a quick demonstration when the knowledge base is small and the matching terms are clear. For larger or semantically complex knowledge bases, vector or other semantic retrieval methods are usually more suitable.